## 8 当前 RNN 模型的弊端

#### 1、为什么要先学习这一节

##### 1.1 这一节的作用是什么
在我们继续学习 GRU 和 LSTM 之前，必须先回答一个非常关键的问题：

>为什么已经有了 RNN，还要再提出 GRU 和 LSTM？

因为在深度学习里，一个新结构通常不是“凭空出现”的，而是为了解决旧结构的某些问题。

所以这一节的核心目标就是：

- 回顾简单 RNN 的工作方式
- 找出它在实际使用中的主要缺点
- 理解这些缺点为什么会推动 GRU 和 LSTM 的出现

##### 1.2 一句话先说明结论
简单来说：

>RNN 虽然可以处理序列数据，但它对“长距离信息”的记忆能力比较弱，  
>在训练时还容易出现梯度消失或梯度爆炸问题，因此在较长序列任务中效果往往不理想。

这就是后面要学习 GRU 和 LSTM 的根本原因。

#### 1、先回顾：简单 RNN 到底在做什么

##### 2.1 RNN 的核心思想
RNN 最核心的特点是：

>当前时刻的隐藏状态，不仅依赖当前输入，还依赖上一个时刻的隐藏状态。

也就是说：

>当前信息 + 过去记忆 → 形成新的记忆

如果写成公式，就是：

$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$

其中：

- $x_t$：当前时刻输入
- $h_{t-1}$：上一个时刻的隐藏状态
- $h_t$：当前时刻隐藏状态

这个结构的直觉很好理解：

模型在读序列时，不是每个词都独立看，而是会把“前面已经读过的信息”带到后面去。

##### 2.2 它为什么看起来很合理
比如一句话：

`“I grew up in France … I speak fluent French.”`

当模型读到后面的 `“French”` 时，如果它还记得前面的 `“France”`，那么就更容易理解这句话。

所以从设计思想上看，RNN 的确是一个很自然的序列模型：

- 它有“时间顺序”
- 它有“状态传递”
- 它可以理论上记住前文信息

这也是为什么 RNN 在早期是处理序列任务的重要模型。


#### 3、长期依赖问题是什么

##### 3.1 什么叫“长期依赖”
所谓长期依赖，指的是：

模型在当前时刻做判断时，需要依赖很久之前出现的信息。

例如：

`“The movie, although slow at the beginning and filled with many unrelated scenes, was absolutely wonderful in the end.”`

**如果模型最后要判断情感，它可能需要综合整个句子的信息，而不是只看最后几个词。**

再例如经典例子：

`“The flowers in the garden near the old house … are beautiful.”`

**这里后面的 `“are”` 要和前面较远位置的 `“flowers”` 对应，而不是被中间很长的修饰部分干扰。**

##### 3.2 简单 RNN 在这里的问题
简单 RNN 理论上可以把前面信息一路传到后面。

但实际上：

>距离一长，前面的关键信息往往已经在状态传递中被弱化甚至丢失了。

所以它容易出现：

- 前文有用信息记不住
- 后面预测过于依赖最近输入
- 长句中的远距离关系捕捉不好

##### 3.3 为什么这会很严重
因为自然语言里，很多重要信息并不总是出现在当前位置附近。

比如：

- 主语和谓语可能隔很远
- 否定词可能出现在前面
- 情感转折可能在中间
- 句子的真正重点可能出现在末尾，但需要前文辅助理解

如果模型只能“看近处”，不能“记远处”，那它对复杂句子的理解就会很差。

#### 四、简单 RNN 的第一个问题：长期记忆能力弱 ❌

##### 4.1 表面上它能记住过去，实际上记不牢
虽然 RNN 的隐藏状态会一层一层往后传：

$h_1 \rightarrow h_2 \rightarrow h_3 \rightarrow \dots \rightarrow h_t$

但问题是：

>这个“记忆”并不是无限稳定的。随着时间步越来越多，前面信息对后面的影响会越来越弱。

也就是说，RNN 并不是不能传递过去信息，而是：

- 近一点的信息，通常还能保留
- 远一点的信息，往往会逐渐丢失

##### 4.2 为什么会这样
因为每个时间步都会做一次新的状态更新：

$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$

注意这里有一个关键点：

>新的隐藏状态 $h_t$ 会不断“覆盖”和“混合”旧的信息。

也就是说，随着时间不断推进：

- 旧信息会不断被新输入冲淡
- 隐藏状态中的内容会不断变化
- 很久以前的信息可能在多次更新后几乎消失

##### 4.3 一个很形象的比喻
假设你要复述一句很长的话，规则是：

每听到一个新词，只允许你在脑子里保留一个“压缩后的摘要”

后面的每个词都要基于这个摘要继续更新

那么短句子还好，一旦句子很长，你最后脑子里能清楚记住的，通常只剩最近的一部分信息。

这就是简单 RNN 的长期记忆问题。

#### 5、简单 RNN 的第二个问题：梯度消失 ❌

##### 5.1 先从反向传播角度理解
我们前面已经学习过：

>RNN 的训练不是普通的单层反向传播，而是 BPTT（Backpropagation Through Time），也就是沿着时间维度展开后再反向传播。

这意味着：

当前时刻的损失，反向传播时要一层一层往前传回去：

$L \rightarrow h_t \rightarrow h_{t-1} \rightarrow h_{t-2} \rightarrow \dots \rightarrow h_1$

问题就出在这里。

##### 5.2 梯度为什么会越来越小
在反向传播过程中，梯度需要经过很多次链式相乘。

而这些乘积里通常包含：

- 权重矩阵
- 激活函数的导数

对于简单 RNN 来说，激活函数经常是 `tanh` 或 `sigmoid`，而这两类函数的导数通常都不大，最大也有限。

所以当梯度经过很多时间步连续相乘后，就会变成：

>越来越小，最后接近 $0$。

这就叫做 梯度消失（Vanishing Gradient）。

##### 5.3 梯度消失带来的直接后果
一旦梯度传不到前面的时间步，就会出现：

- 前面时间步对应的参数几乎得不到有效更新
- 模型学不会“很久以前的信息该如何影响现在”
- 训练虽然在进行，但长期依赖学不出来

换句话说：

>不是模型结构上完全不能传，而是训练过程中学不到。

这点非常重要。

##### 5.4 梯度消失会让 RNN 学不会长距离依赖⚠️
训练的本质是让参数学会“保留重要信息”

RNN 能不能记住长期信息，不只是看结构，还要看训练能不能把参数调到合适的位置。

而参数更新依赖梯度。

如果前面时间步的梯度几乎为 $0$，那么模型就很难学会：

- 哪些旧信息应该保留
- 哪些旧信息对最终结果很重要
- 如何让远处信息持续影响后面状态

本质就是：

>长期依赖问题，在训练层面常常表现为梯度消失问题。

也就是说：

- 结构上：信息传递链条太长
- 训练上：梯度回传链条太长
- 最终结果：远距离信息难学、难保留、难利用

#### 6、简单 RNN 的第三个问题：梯度爆炸 ❌

##### 6.1 梯度不一定总是变小，也可能变大
刚才我们说的是梯度消失。  
但链式相乘还有另一种情况：

>如果某些项大于 $1$，连续相乘后，梯度就可能越来越大，最终变得非常大。

这就是 梯度爆炸（Exploding Gradient）

##### 6.2 梯度爆炸会导致什么
梯度过大时，训练会出现：

- 参数更新幅度异常大
- loss 剧烈震荡
- 模型训练不稳定
- 甚至出现 `nan`

>也就是说，模型不是学得慢，而是直接学崩了。

##### 6.3 和梯度消失相比，它们的区别是什么
梯度消失：前面学不到，模型记不住远处信息

梯度爆炸：训练不稳定，参数更新失控

两者都会影响简单 RNN 的训练效果，只是表现形式不同。


#### 7、简单 RNN 的第四个问题：隐藏状态过于单一 ❌

##### 7.1 简单 RNN 的记忆载体只有一个 $h_t$
在简单 RNN 中，所有历史信息都被压缩到当前隐藏状态 $h_t$ 里。

也就是说：

>模型只有一个统一的“记忆容器”。

这个容器必须同时承担很多任务：

- 记住前文
- 融合当前输入
- 为下一时刻提供状态
- 为输出提供特征

##### 7.2 这会导致什么问题
当序列比较复杂时，这一个隐藏状态就会很吃力。

因为它必须在有限维度里同时存放：

- 近处信息
- 远处信息
- 重要信息
- 次要信息
- 当前上下文特征

这会导致信息表达能力受限。

##### 7.3 一个直观比喻
你可以把简单 RNN 想成：

只有一个小背包的人在长途旅行。

一路上所有东西都要往这个背包里装：

- 重要文件
- 衣服
- 食物
- 水
- 纪念品

背包空间有限，新的东西不断加入，旧的东西就容易被挤掉。

这就是简单 RNN 只有单一隐藏状态时的局限。

#### 8、简单 RNN 的第五个问题：对长序列任务不友好 ❌

##### 8.1 序列越长，问题越明显
在短序列任务中，简单 RNN 有时还能工作得不错。

但当序列变长后，前面的问题会被放大：

- 远距离信息更难保留
- 梯度传播路径更长
- 训练更困难
- 性能更容易下降

##### 8.2 这在自然语言中非常常见
真实文本任务里，经常会遇到：

- 长句子
- 多个从句
- 上下文跨度很大
- 句子之间存在依赖关系

例如：

- 文本分类
- 机器翻译
- 语言建模
- 问答系统

这些任务都可能要求模型处理较长的上下文。

而简单 RNN 一旦序列太长，通常就显得力不从心。

#### 9、一个经典例子：为什么简单 RNN 容易忘记关键信息

##### 9.1 例子说明
看这句话：

`“I grew up in France, and after many years of traveling around the world, learning different languages, and meeting many people, I can still speak fluent French.”`

当模型读到最后的 `“French”` 时，真正有帮助的信息可能是很早以前出现的 `“France”`。

##### 9.2 简单 RNN 面临的困难
从 `“France”` 到最后的 `“French”` 中间隔了很多词。

那么：

- 前面的信息需要经过很多时间步传到后面
- 反向传播时，损失也要跨很多时间步回传

这时就很容易出现前文记忆衰减和梯度消失

结果就是：

>模型可能更依赖附近词，而不能很好利用远处的 `“France”`。

##### 9.3 这就是后续改进模型要解决的问题
所以后面的 GRU 和 LSTM，本质上不是推翻 RNN，而是在想办法解决：

如何让重要信息在长时间跨度中更稳定地保留下来。

#### 10、为什么需要 GRU 和 LSTM

##### 10.1 先给出核心答案
因为简单 RNN 存在以下典型问题：

- 长期依赖难以建模
- 梯度消失严重
- 梯度爆炸风险存在
- 单一隐藏状态表达能力有限
- 长序列训练效果较差

>所以研究者提出了更复杂的循环结构，希望解决这些问题。

##### 10.2 它们改进的核心方向是什么
GRU 和 LSTM 的核心思路都不是“取消循环”，而是：

在循环结构中加入更精细的控制机制，让模型学会：

- 什么信息该保留 ✅
- 什么信息该遗忘 ✅
- 什么信息该更新 ✅
- 什么信息应该更稳定地跨时间传播 ✅

##### 10.3 为什么这很重要
因为简单 RNN 的问题，不是完全没有记忆，而是：

它不会精细地管理记忆。

GRU 和 LSTM 的价值就在这里：

它们让“记忆”不再只是被动更新，而是变成一种可控制的状态流动机制。